# DBSCAN — Deteksi Anomali Harga Komoditas Pangan
**Dataset:** WFP Food Prices Indonesia · **Periode:** Juli 2016 – Januari 2026  
**Parameter final:** eps = 0.18 · minPts = 20 · Silhouette Score = 0.51

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

DATA_PATH = r'C:\Users\Pongo\Downloads\datafoodxlsx.xlsx'
print('Library siap.')

## 2. Load dan Preprocessing Data

In [ ]:
# --- 2.1 Load ---
df = pd.read_excel(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()
df = df[['date', 'price', 'commodity']].copy()

# --- 2.2 Kelompokkan 27+ varian ke 8 komoditas bersih ---
commodity_map = {
    r'rice':                   'Rice',
    r'egg':                    'Eggs',
    r'meat|chicken|beef|pork|mutton|poultry': 'Meat',
    r'sugar':                  'Sugar',
    r'oil|fat':                'Oil',
    r'garlic':                 'Garlic',
    r'onion|shallot':          'Onions',
    r'chili|pepper|chilli':    'Chili',
}

def map_commodity(name):
    name_lower = str(name).lower()
    for pattern, group in commodity_map.items():
        if re.search(pattern, name_lower):
            return group
    return None

df['commodity_clean'] = df['commodity'].apply(map_commodity)
df = df[df['commodity_clean'].notna()].copy()

# --- 2.3 Parse tanggal (Excel serial) ---
df['date'] = pd.to_numeric(df['date'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], origin='1899-12-30', unit='D', errors='coerce')
df = df[df['date'].notna()].copy()

# --- 2.4 Bersihkan harga ---
df['price'] = (df['price'].astype(str)
               .str.replace(',', '', regex=False)
               .str.strip())
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price']).reset_index(drop=True)

print(f'Total baris : {len(df):,}')
print(f'Rentang     : {df["date"].min().date()} – {df["date"].max().date()}')
print(f'Komoditas   : {sorted(df["commodity_clean"].unique())}')

In [ ]:
# --- 2.5 Statistik deskriptif (Tabel 3.1) ---
stats = df.groupby('commodity_clean')['price'].agg(['min', 'max', 'mean', 'std'])
stats.columns = ['Min (Rp/kg)', 'Maks (Rp/kg)', 'Rata-rata', 'Std. Deviasi']
print('=== Tabel 3.1: Statistik Deskriptif ===')
print(stats.round(0).astype(int).to_string())

In [ ]:
# --- 2.6 Feature engineering ---
df['date_ordinal'] = df['date'].map(lambda x: x.toordinal())

scaler = StandardScaler()
scaled = scaler.fit_transform(df[['date_ordinal', 'price']])
scaled_df = pd.DataFrame(scaled, columns=['date_scaled', 'price_scaled'])

df_onehot = pd.get_dummies(df['commodity_clean'], prefix='commodity')
X = pd.concat([scaled_df, df_onehot], axis=1)

print(f'Dimensi fitur: {X.shape[1]}  (2 numerik + {df_onehot.shape[1]} one-hot)')
print(f'Kolom        : {list(X.columns)}')

## 3. Penentuan Parameter — Tahap 1: K-Distance Graph

In [ ]:
# k = minPts = 2 × dim = 2 × 10 = 20
k = 20
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=min(20000, len(X)), replace=False)
X_sample = X.values[sample_idx]

nn = NearestNeighbors(n_neighbors=k, n_jobs=-1)
nn.fit(X_sample)
dists, _ = nn.kneighbors(X_sample)
k_distances = np.sort(dists[:, -1])

plt.figure(figsize=(9, 4))
plt.plot(k_distances, color='steelblue', linewidth=1.2)
plt.axhline(0.44, color='red', linestyle='--', linewidth=1, label='Titik siku ≈ 0.44 (batas atas)')
plt.axhline(0.18, color='green', linestyle='--', linewidth=1, label='ε final = 0.18')
plt.title(f'Grafik {k}-Distance (sampel 20.000 titik)', fontsize=12)
plt.xlabel('Data diurutkan')
plt.ylabel('Jarak ke tetangga ke-20')
plt.legend()
plt.tight_layout()
plt.savefig(r'C:\Users\Pongo\Downloads\fig_kdistance.png', dpi=150)
plt.show()
print('Gambar 4.1 disimpan: fig_kdistance.png')

## 4. Penentuan Parameter — Tahap 2: Uji Sensitivitas Epsilon

In [ ]:
eps_values = [0.10, 0.14, 0.18, 0.22, 0.26, 0.30, 0.44]
MIN_PTS = 20

np.random.seed(42)
sil_idx = np.random.choice(len(X), size=10000, replace=False)

results = []
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=MIN_PTS, n_jobs=-1)
    lbl = db.fit_predict(X.values)
    n_clusters = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_noise    = (lbl == -1).sum()
    pct_noise  = n_noise / len(lbl) * 100

    # Silhouette hanya pada titik dalam cluster (bukan noise)
    mask_in = lbl[sil_idx] != -1
    if mask_in.sum() > 1 and n_clusters > 1:
        sil = silhouette_score(X.values[sil_idx][mask_in], lbl[sil_idx][mask_in])
    else:
        sil = np.nan

    results.append({'eps': eps, 'n_clusters': n_clusters,
                    'n_noise': n_noise, 'pct_noise': pct_noise, 'silhouette': sil})
    print(f'eps={eps:.2f}  cluster={n_clusters:3d}  noise={n_noise:5d} ({pct_noise:.2f}%)  sil={sil:.2f}')

df_sens = pd.DataFrame(results)
print('\n=== Tabel 4.1: Uji Sensitivitas ===')
print(df_sens.to_string(index=False))

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(df_sens['eps'], df_sens['silhouette'], 'b-o', label='Silhouette Score')
ax2.plot(df_sens['eps'], df_sens['pct_noise'], 'r--s', label='% Noise')

ax1.axvline(0.18, color='green', linestyle=':', linewidth=1.5, label='ε terpilih = 0.18')
ax1.set_xlabel('Epsilon')
ax1.set_ylabel('Silhouette Score', color='b')
ax2.set_ylabel('% Noise Point', color='r')
ax1.set_title('Uji Sensitivitas: Silhouette Score vs % Noise')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.savefig(r'C:\Users\Pongo\Downloads\fig_sensitivity.png', dpi=150)
plt.show()
print('Gambar 4.2 disimpan: fig_sensitivity.png')

## 5. DBSCAN Final (eps = 0.18, minPts = 20)

In [ ]:
print('Menjalankan DBSCAN final...')
db_final = DBSCAN(eps=0.18, min_samples=20, n_jobs=-1)
labels = db_final.fit_predict(X.values)
df['cluster'] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

print(f'Jumlah cluster : {n_clusters}')
print(f'Noise (anomali): {n_noise} ({n_noise/len(df)*100:.2f}%)')
print(f'Total          : {len(df):,}')

# --- Distribusi per cluster (untuk Tabel 4.2) ---
cluster_counts = pd.Series(labels).value_counts().sort_index()
print('\n=== Distribusi Per Cluster (data untuk Tabel 4.2) ===')
for c, cnt in cluster_counts.items():
    label = 'Noise' if c == -1 else f'Cluster {c}'
    print(f'{label:12s}: {cnt:7,} ({cnt/len(df)*100:.2f}%)')
print(f'\nVERIFIKASI TOTAL: {cluster_counts.sum():,}')

In [ ]:
# --- Komoditas dominan per cluster ---
print('=== Komoditas Dominan Per Cluster ===')
for c in sorted(set(labels)):
    subset = df[df['cluster'] == c]
    dom = subset['commodity_clean'].value_counts().index[0]
    n   = len(subset)
    label = 'Noise(-1)' if c == -1 else f'Cluster {c:2d}'
    print(f'{label}: {n:7,}  → dominan: {dom}')

## 6. Visualisasi Hasil Clustering

In [ ]:
# --- Gambar 4.3: Scatter date_scaled vs price_scaled ---
normal_mask  = df['cluster'] != -1
anomali_mask = df['cluster'] == -1

plt.figure(figsize=(12, 5))
sc = plt.scatter(
    scaled_df.loc[normal_mask, 'date_scaled'],
    scaled_df.loc[normal_mask, 'price_scaled'],
    c=df.loc[normal_mask, 'cluster'],
    cmap='tab20', s=2, alpha=0.4, label='Cluster'
)
plt.scatter(
    scaled_df.loc[anomali_mask, 'date_scaled'],
    scaled_df.loc[anomali_mask, 'price_scaled'],
    c='red', s=30, marker='x', label=f'Noise/Anomali (n={n_noise})', zorder=5
)
plt.colorbar(sc, label='Cluster ID')
plt.xlabel('date_scaled')
plt.ylabel('price_scaled')
plt.title('Gambar 4.3: Hasil DBSCAN (eps=0.18, minPts=20)')
plt.legend(markerscale=3)
plt.tight_layout()
plt.savefig(r'C:\Users\Pongo\Downloads\fig_scatter_dbscan.png', dpi=150)
plt.show()
print('Gambar 4.3 disimpan.')

## 7. Analisis Anomali (Noise Points)

In [ ]:
anomali_df = df[df['cluster'] == -1].copy()

print('=== Tabel 4.3: Distribusi Anomali Per Komoditas ===')
dist = anomali_df.groupby('commodity_clean').agg(
    jumlah=('price', 'count'),
    harga_maks=('price', 'max')
).sort_values('jumlah', ascending=False)
dist['persen'] = (dist['jumlah'] / n_noise * 100).round(1)
print(dist.to_string())

print('\n=== Distribusi Temporal (per tahun) ===')
print(anomali_df['date'].dt.year.value_counts().sort_index())

print('\n=== 5 Anomali Harga Tertinggi ===')
top5 = anomali_df.nlargest(5, 'price')[['date', 'commodity_clean', 'price']]
print(top5.to_string(index=False))

In [ ]:
# --- Gambar 4.4: Distribusi temporal noise per bulan ---
anomali_df['year_month'] = anomali_df['date'].dt.to_period('M')
monthly = anomali_df.groupby('year_month').size()

plt.figure(figsize=(13, 4))
monthly.plot(kind='bar', color='crimson', width=0.8)
plt.title('Gambar 4.4: Distribusi Temporal Noise Point per Bulan')
plt.xlabel('Bulan')
plt.ylabel('Jumlah Anomali')
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()
plt.savefig(r'C:\Users\Pongo\Downloads\fig_temporal_noise.png', dpi=150)
plt.show()
print('Gambar 4.4 disimpan.')

## 8. Perbandingan dengan KMeans

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=8, random_state=42, n_init=10)
km_labels = km.fit_predict(X.values)

# Silhouette KMeans
np.random.seed(42)
s_idx = np.random.choice(len(X), 10000, replace=False)
sil_km = silhouette_score(X.values[s_idx], km_labels[s_idx])

# Silhouette DBSCAN (cluster only)
mask_db = labels[s_idx] != -1
sil_db = silhouette_score(X.values[s_idx][mask_db], labels[s_idx][mask_db])

print(f'Silhouette KMeans (k=8) : {sil_km:.3f}')
print(f'Silhouette DBSCAN       : {sil_db:.3f}')
print(f'\nKMeans tidak menghasilkan noise point secara eksplisit.')
print(f'DBSCAN menandai {n_noise} titik sebagai anomali dalam satu langkah.')

## Ringkasan Output untuk Laporan

| Metrik | Nilai |
|--------|-------|
| Total observasi | 208.136 |
| Epsilon | 0,18 |
| minPts | 20 |
| Jumlah cluster | 19 |
| Noise point (anomali) | 276 (0,13%) |
| Silhouette Score | 0,51 |
| Komoditas anomali terbanyak | Meat (113), Chili (110) |